In [6]:
import numpy as np
import pandas as pd

# Parâmetros do Ambiente
states  = ['baixa', 'normal', 'alta']
actions = ['reduzir', 'manter', 'aumentar']
S, A = len(states), len(actions)

# Parâmetros do Algoritmo
alpha, gamma = 0.1, 0.95
eps_i, eps_f = 0.3, 0.02
episodes, steps = 5000, 20
rng = np.random.default_rng(42)

# Funções de Transição
def transition_probs(s, a):
    sname, aname = states[s], actions[a]

    if sname == 'alta':
        if aname == 'aumentar': p = [0.10, 0.75, 0.15]
        elif aname == 'manter': p = [0.05, 0.45, 0.50]
        else:                   p = [0.05, 0.20, 0.75]

    elif sname == 'baixa':
        if aname == 'reduzir':  p = [0.10, 0.80, 0.10]
        elif aname == 'manter': p = [0.45, 0.40, 0.15]
        else:                   p = [0.75, 0.15, 0.10]

    else:
        if aname == 'manter':   p = [0.10, 0.80, 0.10]
        elif aname == 'aumentar': p = [0.25, 0.60, 0.15]
        else:                   p = [0.10, 0.65, 0.25]

    return np.array(p)

# Funções de Recompensa
def reward_clinical(next_s):
    if states[next_s] == 'normal': return 10
    elif states[next_s] == 'baixa': return -12
    else: return -5

def reward_action_cost(a):
    if actions[a] == 'manter': return 0.0
    else: return -0.5

def reward_total(next_s, a):
    return reward_clinical(next_s) + reward_action_cost(a)

# Algoritmo Q-Learning
Q = np.zeros((S, A))

def select_action(s, epsilon):
    if rng.random() < epsilon:
        return rng.integers(A)
    else:
        return np.argmax(Q[s, :])

print("Iniciando Treinamento Q-Learning...")

for episode in range(episodes):
    epsilon = eps_i - (eps_i - eps_f) * (episode / episodes)
    s = rng.choice(S)

    for step in range(steps):
        a = select_action(s, epsilon)
        probs = transition_probs(s, a)
        next_s = rng.choice(S, p=probs)
        R = reward_total(next_s, a)

        max_Q_next = np.max(Q[next_s, :])
        td_error = (R + gamma * max_Q_next) - Q[s, a]
        Q[s, a] += alpha * td_error

        s = next_s

print("Treinamento concluído.")

# Resultados
Q_table_df = pd.DataFrame(Q, index=states, columns=actions)

print("\nTabela Q Final:")
print(Q_table_df.round(3))

policy = Q_table_df.idxmax(axis=1)
print("\nPolítica Ótima:")
print(policy)

print("\nInterpretação da Política:")
print(f"Estado 'baixa' (Hipotensão): Ação ideal é '{policy['baixa']}'.")
print(f"Estado 'normal': Ação ideal é '{policy['normal']}'.")
print(f"Estado 'alta' (Hipertensão): Ação ideal é '{policy['alta']}'.")

Iniciando Treinamento Q-Learning...
Treinamento concluído.

Tabela Q Final:
        reduzir   manter  aumentar
baixa   117.631  103.669   105.187
normal  114.490  122.563   110.995
alta    106.385  111.540   118.927

Política Ótima:
baixa      reduzir
normal      manter
alta      aumentar
dtype: object

Interpretação da Política:
Estado 'baixa' (Hipotensão): Ação ideal é 'reduzir'.
Estado 'normal': Ação ideal é 'manter'.
Estado 'alta' (Hipertensão): Ação ideal é 'aumentar'.
